# Phase 4 CJPE — V4 Hybrid Model with LoRA

This stage adapts the existing Phase 1 V4 hybrid checkpoint to binary case-outcome prediction. It freezes the pretrained model and trains rank-8 LoRA updates on query/value projections plus a new task head. It does not restart Phase 1 training.

This continues the model-by-model plan: V1 LoRA, V2 LoRMA+, V3 BitFit, and now V4 hybrid + LoRA. The backbone and adapter vary across stages, so scores compare practical candidate systems rather than isolating the effect of one method. The V4 hybrid checkpoint is the model-only artifact; optimizer resume checkpoints are unnecessary.

The notebook uses the same supplied train/dev/test contract, tokenizer, document windows, development selection, and root-level prediction ZIP format. It reads no test labels and makes no score guarantee.

**Reference:** Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models (2021)](https://arxiv.org/abs/2106.09685).

**Run order:** attach the listed Kaggle inputs, restart the session, then use **Run All** from the first cell. The data/tokenizer cell depends on `ASSET`, created in “Inputs and run settings”; running later cells alone after a restart leaves that variable undefined.


## 1. Inputs and run settings

Attach train.json, dev.json, test.json, hybrid_model.pt, and the original bpe_en_25k.json tokenizer to Kaggle once. Select a GPU accelerator. The notebook stops when an input is absent or duplicated. It loads the model-only Phase 1 checkpoint and never resumes Phase 1 training.

In [ ]:
import time

import os,json,math,time,gc,zipfile,random,copy
from pathlib import Path
from collections import Counter
from functools import lru_cache
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.utils.checkpoint import checkpoint
SEED=781; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'; AMP_DTYPE=torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
INPUT_ROOT=Path('/kaggle/input'); OUT=Path('/kaggle/working/phase4_v4_lora'); OUT.mkdir(parents=True,exist_ok=True)
# Long-document strategy: overlapping 512-token windows, spaced across the case.
CHUNK_TOKENS=512; CHUNK_STRIDE=384; MAX_CHUNKS_PER_CASE=8; TRAIN_CHUNKS_PER_CASE=1
BATCH_CASES=8; GRAD_ACCUM_STEPS=2; INFER_CHUNK_BATCH=16
# LoRA settings and optimizer.
LORA_RANK=8; LORA_ALPHA=16; LORA_DROPOUT=0.05; HEAD_DROPOUT=0.10
LORA_LR=1e-4; HEAD_LR=3e-4; WEIGHT_DECAY=0.01; MAX_EPOCHS=4; PATIENCE=2; PROGRESS_EVERY=50
WARMUP_RATIO=0.03; MAX_GRAD_NORM=1.0

def find_one(name):
    found=sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    if len(found)!=1: raise FileNotFoundError(f'Expected one {name}; found {len(found)}: {found[:8]}')
    return found[0]
ASSET={n:find_one(n) for n in ('train.json','dev.json','test.json','hybrid_model.pt','bpe_en_25k.json')}
print('Device:',DEVICE,'GPU:',torch.cuda.get_device_name(0) if USE_AMP else 'none')
for n,p in ASSET.items(): print(n,'->',p)



## 2. Split checks and the V4 tokenizer

Training and development labels must be integer 0/1. Test rows are reduced to `id` and `text`; any extra test fields are discarded. The exact V4 BPE algorithm is preserved, with a bounded cache for repeated words.

**Duplicate-record handling:** the supplied train/dev files contain exact repeated case rows. The loader keeps one copy per ID, removes only rows with identical text and label, and raises an error if an ID has conflicting content or labels.


In [ ]:

def load_cases(path,labelled):
    with Path(path).open(encoding='utf-8') as f: rows=json.load(f)
    if not isinstance(rows,list): raise TypeError(f'{path}: expected JSON list')
    result=[]; seen={}; duplicate_count=0
    for i,r in enumerate(rows):
        if not isinstance(r,dict) or 'id' not in r or 'text' not in r: raise ValueError(f'{path}: row {i} lacks id/text')
        cid=str(r['id']); text=r['text']
        if not isinstance(text,str): raise TypeError(f'{cid}: text must be string')
        label=r.get('label') if labelled else None
        if labelled and (type(label) is not int or label not in (0,1)): raise ValueError(f'{cid}: label must be integer 0/1')
        normalized={'id':cid,'text':text}
        if labelled: normalized['label']=label
        if cid in seen:
            if result[seen[cid]] != normalized:
                raise ValueError(f'{path}: conflicting duplicate id {cid} (text or label differs)')
            duplicate_count+=1
            continue
        r.clear(); r.update(normalized)
        seen[cid]=len(result); result.append(r)
    if duplicate_count: print(f'{path}: removed {duplicate_count} exact duplicate rows; kept {len(result)} unique IDs')
    return result
TRAIN_ROWS=load_cases(ASSET['train.json'],True); DEV_ROWS=load_cases(ASSET['dev.json'],True); TEST_ROWS=load_cases(ASSET['test.json'],False)
sets=[{r['id'] for r in rows} for rows in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)]
if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise ValueError('Split IDs overlap')
print('Split sizes:',*[len(x) for x in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)])
print('Train label counts:',dict(Counter(r['label'] for r in TRAIN_ROWS)))
print('Development label counts:',dict(Counter(r['label'] for r in DEV_ROWS)))
print('Test loader retains id/text only.')

with ASSET['bpe_en_25k.json'].open(encoding='utf-8') as f: bpe=json.load(f)
bpe_itos=bpe['itos']; bpe_merges=bpe['merges']; bpe_stoi={t:i for i,t in enumerate(bpe_itos)}
if bpe['vocab_size']!=25000 or len(bpe_itos)!=25000: raise ValueError('Expected V4 BPE-25k')
merge_to_id={}
for i,(left,right) in enumerate(bpe_merges): merge_to_id[(bpe_stoi[left],bpe_stoi[right])]=261+i
PAD_TOKEN_ID=bpe_stoi['<pad>']; BOS_TOKEN_ID=bpe_stoi['<s>']; EOS_TOKEN_ID=bpe_stoi['</s>']

class ByteLevelBPE25k:
    def __init__(self,itos,stoi,merge_to_id,merges):
        self.itos=itos; self.stoi=stoi; self.merge_to_id=merge_to_id; self.merges=merges; self.vocab_size=len(itos)
        self.pad_token_id=stoi['<pad>']; self.bos_token_id=stoi['<s>']; self.eos_token_id=stoi['</s>']
        self.unk_token_id=stoi['<unk>']; self.word_boundary_id=stoi['<wb>']
    @lru_cache(maxsize=100000)
    def _encode_word(self,word):
        seq=[4+b for b in word.encode('utf-8')]+[self.word_boundary_id]
        while len(seq)>1:
            best_pos=None; best_id=None
            for i in range(len(seq)-1):
                new_id=self.merge_to_id.get((seq[i],seq[i+1]))
                if new_id is not None and (best_id is None or new_id<best_id): best_id=new_id; best_pos=i
            if best_pos is None: break
            seq=seq[:best_pos]+[best_id]+seq[best_pos+2:]
        return seq
    def encode(self,text,add_bos=False,add_eos=False):
        ids=[self.bos_token_id] if add_bos else []
        for word in text.split(): ids.extend(self._encode_word(word))
        if add_eos: ids.append(self.eos_token_id)
        return ids
    def _to_bytes(self,tid):
        tok=self.itos[tid]
        if tok.startswith('<0x') and tok.endswith('>'): return bytes([int(tok[3:-1],16)])
        if tok=='<wb>': return b' '
        if tok in {'<pad>','<unk>','<s>','</s>'}: return b''
        if tok.startswith('<bpe_'):
            j=tid-261
            if 0<=j<len(self.merges):
                a,b=self.merges[j]; return self._to_bytes(self.stoi[a])+self._to_bytes(self.stoi[b])
        return b''
    def decode(self,ids):
        out=bytearray()
        for tid in ids:
            if 0<=int(tid)<self.vocab_size: out.extend(self._to_bytes(int(tid)))
        return bytes(out).decode('utf-8',errors='replace')

tokenizer=ByteLevelBPE25k(bpe_itos,bpe_stoi,merge_to_id,bpe_merges)
probe='The Supreme Court of India has jurisdiction over constitutional matters.'
decoded_probe=tokenizer.decode(tokenizer.encode(probe))
# The final <wb> token decodes to one trailing space by design.
assert decoded_probe.rstrip()==probe, repr(decoded_probe)
print('V4 tokenizer loaded; vocabulary:',tokenizer.vocab_size)


## 3. Rebuild the exact V4 hybrid architecture and strictly load Phase 1 weights

The architecture definition is reused from the existing V4 Phase 3 notebook, which embeds the original hybrid Transformer implementation. Its four global-attention layers and 256-token local window are set before constructing the model. The checkpoint configuration is checked field by field and the state dictionary is loaded strictly before adding any task-specific layers.

In [ ]:
# V4 global/local attention settings from the saved Phase 1 configuration.
GLOBAL_ATTENTION_LAYERS=[0,4,8,11]
LOCAL_ATTN_WINDOW=256

# 9. V4 Hybrid Decoder-Only Transformer
#
# V2 preserves the successful the original hybrid-model components:
# - RMSNorm
# - RoPE
# - Grouped Query Attention (GQA)
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache
#
# New architectural component:
# - Hybrid Global + Local Causal Attention
#
# Global layers:
#   Full causal attention over the complete sequence
#
# Local layers:
#   Causal attention restricted to a fixed local window
#
# The hybrid design is motivated by the course discussion of
# sparse/local attention and hybrid models that interleave
# full and sparse attention layers.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Hybrid Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        attention_type="global",
        local_window=256,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        assert attention_type in {
            "global",
            "local"
        }

        assert local_window > 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.attention_type = attention_type
        self.local_window = local_window

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def _build_attention_mask(
        self,
        query_length,
        key_length,
        query_position_offset,
        device
    ):
        """
        Build a boolean attention mask.

        True  = attention is allowed
        False = attention is blocked

        For global attention:
            causal mask

        For local attention:
            causal + fixed sliding window
        """

        query_positions = (
            query_position_offset
            + torch.arange(
                query_length,
                device=device
            )
        )

        key_positions = torch.arange(
            key_length,
            device=device
        )

        # Causal condition:
        # query at position q can only see keys <= q.
        causal_mask = (
            key_positions.unsqueeze(0)
            <= query_positions.unsqueeze(1)
        )

        if self.attention_type == "global":
            return causal_mask

        # Local causal attention:
        # q can see at most the previous
        # local_window - 1 tokens plus itself.
        local_mask = (
            key_positions.unsqueeze(0)
            >= (
                query_positions.unsqueeze(1)
                - self.local_window
                + 1
            )
        )

        return causal_mask & local_mask

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        # -------------------------------------------------
        # Q, K, V projections
        # -------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        # -------------------------------------------------
        # KV cache information
        # -------------------------------------------------

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # -------------------------------------------------
        # RoPE
        # -------------------------------------------------

        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # -------------------------------------------------
        # Append new K/V to cache
        # -------------------------------------------------

        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # -------------------------------------------------
        # GQA
        # -------------------------------------------------

        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # -------------------------------------------------
        # Hybrid attention mask
        # -------------------------------------------------

        key_length = k.size(-2)

        attn_mask = self._build_attention_mask(
            query_length=T,
            key_length=key_length,
            query_position_offset=past_len,
            device=x.device
        )

        # -------------------------------------------------
        # Scaled dot-product attention
        # -------------------------------------------------

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        # -------------------------------------------------
        # Output projection
        # -------------------------------------------------

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard GELU FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Hybrid Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout,
        layer_index,
        global_attention_layers,
        local_window
    ):
        super().__init__()

        self.layer_index = layer_index

        if layer_index in global_attention_layers:
            attention_type = "global"
        else:
            attention_type = "local"

        self.attention_type = attention_type

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            attention_type=attention_type,
            local_window=local_window,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention + residual.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN + residual.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Lightweight Hybrid Legal Language Model
# ---------------------------------------------------------

class HybridLegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout,
                layer_index=layer_index,
                global_attention_layers=GLOBAL_ATTENTION_LAYERS,
                local_window=LOCAL_ATTN_WINDOW
            )
            for layer_index in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Weight tying retained from V1.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
V4_CONTEXT=1024; N_EMBD=768; N_LAYER=12; N_HEAD=12; N_KV_HEAD=4; FFN_MULTIPLIER=4
ckpt=torch.load(ASSET['hybrid_model.pt'],map_location='cpu',weights_only=False)
if not isinstance(ckpt,dict) or 'model_state_dict' not in ckpt or 'config' not in ckpt:
    raise ValueError('hybrid_model.pt must contain model_state_dict and config')
V4_CONFIG=ckpt['config']
EXPECTED={'vocab_size':25000,'n_embd':N_EMBD,'n_layer':N_LAYER,'n_head':N_HEAD,
          'n_kv_head':N_KV_HEAD,'ffn_multiplier':FFN_MULTIPLIER,'context_length':V4_CONTEXT,
          'local_attention_window':256}
for key,expected in EXPECTED.items():
    if key not in V4_CONFIG or int(V4_CONFIG[key])!=expected:
        raise ValueError(f'V4 checkpoint config mismatch for {key}: {V4_CONFIG.get(key)} vs {expected}')
if list(map(int,V4_CONFIG.get('global_attention_layers',[])))!=GLOBAL_ATTENTION_LAYERS:
    raise ValueError(f"Unexpected V4 global-attention pattern: {V4_CONFIG.get('global_attention_layers')}")
v4=HybridLegalLM(vocab_size=tokenizer.vocab_size,n_embd=N_EMBD,n_layer=N_LAYER,
    n_head=N_HEAD,n_kv_head=N_KV_HEAD,max_seq_len=V4_CONTEXT,ffn_multiplier=FFN_MULTIPLIER,dropout=0.0)
state=ckpt['model_state_dict']
missing,unexpected=v4.load_state_dict(state,strict=True)
assert not missing and not unexpected,(missing,unexpected)
PHASE1_STATE={k:t.detach().cpu() for k,t in state.items()}
PHASE1_PARAMETER_COUNT=sum(p.numel() for p in v4.parameters())
assert PHASE1_PARAMETER_COUNT==94_762_752,f'Unexpected V4 parameter count: {PHASE1_PARAMETER_COUNT:,}'
print('Strict V4 checkpoint load passed; unique parameters:',f'{PHASE1_PARAMETER_COUNT:,}')
print('Checkpoint metadata:',json.dumps(V4_CONFIG,indent=2))
del ckpt,state
gc.collect()

## 4. LoRA on query and value projections

LoRA wraps the V4 query and value projections in every Transformer block with a rank-8, alpha-16 low-rank update. The update's B matrix starts at zero, so adding the adapter initially preserves each original projection. The hybrid global/local attention rule and all original V4 weights remain unchanged and frozen.

In [ ]:

class LoRALinear(nn.Module):
    def __init__(self,base,rank=8,alpha=16,dropout=0.05):
        super().__init__(); self.base=base; self.rank=rank; self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=base.weight.dtype))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        for p in self.base.parameters(): p.requires_grad_(False)
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(self.dropout(x),self.A),self.B)*self.scale

for p in v4.parameters(): p.requires_grad_(False)
for block in v4.blocks:
    block.attention.q_proj=LoRALinear(block.attention.q_proj,LORA_RANK,LORA_ALPHA,LORA_DROPOUT)
    block.attention.v_proj=LoRALinear(block.attention.v_proj,LORA_RANK,LORA_ALPHA,LORA_DROPOUT)

class CaseClassifier(nn.Module):
    def __init__(self,backbone):
        super().__init__(); self.backbone=backbone
        self.head=nn.Sequential(nn.Dropout(HEAD_DROPOUT),nn.Linear(N_EMBD,2))
    def forward(self,input_ids,attention_mask):
        x=self.backbone.token_embedding(input_ids)
        for block in self.backbone.blocks:
            x,_=block(x,past_key_value=None,use_cache=False)
        x=self.backbone.final_norm(x)
        last=(attention_mask.sum(1)-1).clamp(min=0)
        pooled=x[torch.arange(x.size(0),device=x.device),last]
        return self.head(pooled)

model=CaseClassifier(v4).to(DEVICE)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)')
assert trainable<1_000_000



## 5. Long judgments as spaced, overlapping windows

V4 accepts up to 1,024 tokens, but full judgments can be far longer. We use 512-token windows with 128-token overlap, then choose up to eight windows spread from the beginning to the end of each case. During training, one of those windows is rotated per case across epochs to control compute. Development and test scores average the logits from all selected windows. This gives broad coverage but is still a capped view: no claim is made that every token is seen in every pass. The beginning/middle/end diagnostic later shows whether this window aggregation matters.


In [ ]:

def select_windows(text):
    ids=tokenizer.encode(text,add_bos=True)
    if not ids: ids=[BOS_TOKEN_ID]
    if len(ids)<=CHUNK_TOKENS: starts=[0]
    else:
        all_starts=list(range(0,len(ids)-CHUNK_TOKENS+1,CHUNK_STRIDE))
        if all_starts[-1]!=len(ids)-CHUNK_TOKENS: all_starts.append(len(ids)-CHUNK_TOKENS)
        take=min(MAX_CHUNKS_PER_CASE,len(all_starts))
        starts=sorted(set(np.linspace(0,len(all_starts)-1,take).round().astype(int).tolist()))
        starts=[all_starts[i] for i in starts]
    chunks=[]
    for start in starts:
        x=ids[start:start+CHUNK_TOKENS]
        chunks.append(x+[PAD_TOKEN_ID]*(CHUNK_TOKENS-len(x)))
    return np.asarray(chunks,dtype=np.int32)

def encode_rows(rows,name):
    counts=[]; chunks=[]
    for i,row in enumerate(rows):
        c=select_windows(row['text']); chunks.append(c); counts.append(len(c))
        if (i+1)%5000==0: print(name,i+1,'/',len(rows))
    print(name,'windows:',sum(counts),'documents:',len(rows),'median windows:',int(np.median(counts)))
    return chunks
TRAIN_WINDOWS=encode_rows(TRAIN_ROWS,'train'); DEV_WINDOWS=encode_rows(DEV_ROWS,'dev'); TEST_WINDOWS=encode_rows(TEST_ROWS,'test')

class TrainDataset(Dataset):
    def __init__(self,rows,windows): self.rows=rows; self.windows=windows; self.epoch=0
    def set_epoch(self,e): self.epoch=e
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        ws=self.windows[i]; idx=(self.epoch+i)%len(ws)
        x=ws[idx]; mask=(x!=PAD_TOKEN_ID)
        return torch.tensor(x,dtype=torch.long),torch.tensor(mask,dtype=torch.bool),self.rows[i]['label']
class EvalDataset(Dataset):
    def __init__(self,rows,windows): self.rows=rows; self.windows=windows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.windows[i],self.rows[i]['id'],self.rows[i].get('label')

train_ds=TrainDataset(TRAIN_ROWS,TRAIN_WINDOWS)
dev_ds=EvalDataset(DEV_ROWS,DEV_WINDOWS); test_ds=EvalDataset(TEST_ROWS,TEST_WINDOWS)
train_loader=DataLoader(train_ds,batch_size=BATCH_CASES,shuffle=True,num_workers=0,pin_memory=USE_AMP)



## 6. Train only the new task head and LoRA matrices

The Phase 1 checkpoint is frozen. We use weighted cross-entropy based only on the training labels, AdamW, a smaller learning rate for the low-rank updates than the fresh head, and early stopping on development macro-F1. The weight is set from training label counts so the less frequent outcome contributes more. We log both unweighted macro-F1 and accuracy; accuracy is secondary because of the class imbalance.


In [ ]:

if DEVICE.type!='cuda': raise RuntimeError('Select a Kaggle GPU accelerator before training this 95M-parameter model.')
def scores(y_true,y_pred):
    out={}; f1s=[]
    for c in (0,1):
        tp=sum(a==c and b==c for a,b in zip(y_true,y_pred)); fp=sum(a!=c and b==c for a,b in zip(y_true,y_pred)); fn=sum(a==c and b!=c for a,b in zip(y_true,y_pred))
        p=tp/(tp+fp) if tp+fp else 0.; r=tp/(tp+fn) if tp+fn else 0.; f=2*p*r/(p+r) if p+r else 0.
        out[str(c)]={'precision':p,'recall':r,'f1':f,'support':sum(a==c for a in y_true)}; f1s.append(f)
    out['macro_f1']=sum(f1s)/2; out['accuracy']=sum(a==b for a,b in zip(y_true,y_pred))/max(1,len(y_true))
    return out

@torch.no_grad()
def predict_windows(rows,windows,mode='all',threshold=0.5):
    model.eval(); selected=[]; owners=[]
    for i,ws in enumerate(windows):
        if mode=='begin': chosen=ws[:1]
        elif mode=='middle': chosen=ws[(len(ws)-1)//2:(len(ws)-1)//2+1]
        elif mode=='end': chosen=ws[-1:]
        else: chosen=ws
        selected.extend(chosen); owners.extend([i]*len(chosen))
    flat=np.stack(selected,axis=0)
    if flat.ndim!=2: raise ValueError(f"Expected prediction windows with shape [windows, tokens], got {flat.shape}")
    all_logits=[]
    infer_started=time.time(); infer_total=math.ceil(len(flat)/INFER_CHUNK_BATCH)
    for batch_idx,j in enumerate(range(0,len(flat),INFER_CHUNK_BATCH),start=1):
        ids=torch.as_tensor(flat[j:j+INFER_CHUNK_BATCH],dtype=torch.long,device=DEVICE); mask=ids.ne(PAD_TOKEN_ID)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP): z=model(ids,mask)
        all_logits.append(z.float().cpu().numpy())
        if batch_idx==1 or batch_idx%PROGRESS_EVERY==0 or batch_idx==infer_total:
            elapsed=time.time()-infer_started; rate=batch_idx/max(elapsed,1e-6); eta=(infer_total-batch_idx)/max(rate,1e-6)
            print(f"Predict {mode}: batch {batch_idx}/{infer_total} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
    window_logits=np.concatenate(all_logits,axis=0); logits_sum=np.zeros((len(rows),2),dtype=np.float64); counts=np.zeros(len(rows),dtype=np.int64)
    np.add.at(logits_sum,np.asarray(owners),window_logits); np.add.at(counts,np.asarray(owners),1)
    logits=(logits_sum/counts[:,None]).astype(np.float32); probs=torch.softmax(torch.tensor(logits),dim=-1).numpy()[:,1]
    return logits,probs,(probs>=threshold).astype(int).tolist()

counts=np.bincount([r['label'] for r in TRAIN_ROWS],minlength=2)
class_weights=torch.tensor(len(TRAIN_ROWS)/(2*np.maximum(counts,1)),dtype=torch.float,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=class_weights)
opt=torch.optim.AdamW([
    {'params':[p for n,p in model.named_parameters() if p.requires_grad and '.A' in n or p.requires_grad and '.B' in n],'lr':LORA_LR},
    {'params':[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('head.')],'lr':HEAD_LR}],weight_decay=WEIGHT_DECAY)
total_updates=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)*MAX_EPOCHS; warmup=max(1,int(total_updates*WARMUP_RATIO))
def lr_scale(step): return min((step+1)/warmup,1.) if step<warmup else max(0.,(total_updates-step)/(total_updates-warmup))
scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lr_scale)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
history=[]; best_f1=-1.; best_loss=float('inf'); best_epoch=0; stale=0; update=0
for epoch in range(MAX_EPOCHS):
    train_ds.set_epoch(epoch); model.train(); opt.zero_grad(set_to_none=True); loss_sum=0.; seen=0; pending=0; epoch_started=time.time()
    print(f"Starting epoch {epoch+1}/{MAX_EPOCHS}: {len(train_loader)} batches",flush=True)
    for step,(ids,mask,y) in enumerate(train_loader):
        ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True); y=torch.as_tensor(y,device=DEVICE,dtype=torch.long)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP): logits=model(ids,mask); loss=criterion(logits,y)
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward(); pending+=1; loss_sum+=float(loss.detach())*len(y); seen+=len(y)
        if step==0 or (step+1)%PROGRESS_EVERY==0 or step+1==len(train_loader):
            elapsed=time.time()-epoch_started; rate=(step+1)/max(elapsed,1e-6); eta=(len(train_loader)-step-1)/max(rate,1e-6)
            print(f"Train epoch {epoch+1}/{MAX_EPOCHS}: batch {step+1}/{len(train_loader)} | running loss {loss_sum/max(seen,1):.4f} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min",flush=True)
        if pending==GRAD_ACCUM_STEPS or step==len(train_loader)-1:
            if pending<GRAD_ACCUM_STEPS:
                correction=GRAD_ACCUM_STEPS/pending
                for p in model.parameters():
                    if p.grad is not None: p.grad.mul_(correction)
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); scheduler.step(); update+=1; pending=0
    dev_logits,dev_probs,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
    dev_y=[r['label'] for r in DEV_ROWS]; dev_metrics=scores(dev_y,dev_pred)
    record={'epoch':epoch+1,'train_loss':loss_sum/max(seen,1),'dev':dev_metrics,'lr_lora':opt.param_groups[0]['lr'],'lr_head':opt.param_groups[1]['lr']}
    history.append(record); print(f"Epoch {epoch+1}/{MAX_EPOCHS} | train loss {record['train_loss']:.4f} | dev macro-F1 {dev_metrics['macro_f1']:.4f} | accuracy {dev_metrics['accuracy']:.4f}")
    if dev_metrics['macro_f1']>best_f1 or (dev_metrics['macro_f1']==best_f1 and record['train_loss']<best_loss):
        best_f1=dev_metrics['macro_f1']; best_loss=record['train_loss']; best_epoch=epoch+1; stale=0
        torch.save({'adapter_state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items() if k.startswith('backbone.blocks.') and ('.q_proj.A' in k or '.q_proj.B' in k or '.v_proj.A' in k or '.v_proj.B' in k) or k.startswith('head.')},'epoch':best_epoch,'dev_macro_f1':best_f1,'config':{'rank':LORA_RANK,'alpha':LORA_ALPHA,'chunk_tokens':CHUNK_TOKENS,'max_chunks':MAX_CHUNKS_PER_CASE}},OUT/'v4_lora_best.pt')
    else: stale+=1
    (OUT/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
    if stale>=PATIENCE: print('Early stopping: development macro-F1 did not improve.'); break
print('Best epoch:',best_epoch,'best raw dev macro-F1:',best_f1)



## 7. Restore best adapter, inspect context ablations, and calibrate on development only

We report beginning-only, middle-only, end-only, and all-window development metrics. These are diagnostics, not four new submissions. After selecting the best epoch by raw development macro-F1 at threshold 0.5, a simple threshold sweep is performed on development probabilities only. The chosen threshold is then frozen for test prediction.


In [ ]:

best=torch.load(OUT/'v4_lora_best.pt',map_location=DEVICE,weights_only=False)
own=model.state_dict(); own.update(best['adapter_state_dict']); model.load_state_dict(own,strict=True); model.eval()
dev_y=[r['label'] for r in DEV_ROWS]; ablations={}
for mode in ('begin','middle','end','all'):
    _,prob,pred=predict_windows(DEV_ROWS,DEV_WINDOWS,mode=mode)
    ablations[mode]=scores(dev_y,pred)
    print(mode,ablations[mode])
_,dev_prob,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
thresholds=np.arange(0.30,0.701,0.01); threshold_results=[]
for t in thresholds:
    m=scores(dev_y,(dev_prob>=t).astype(int).tolist()); threshold_results.append((m['macro_f1'],t,m))
_,best_threshold,best_calibrated=max(threshold_results,key=lambda x:(x[0],-abs(x[1]-0.5)))
print('Development-selected threshold:',round(float(best_threshold),2),'metrics:',best_calibrated)
summary={'best_epoch':best_epoch,'best_checkpoint_selection_macro_f1':best_f1,'development_threshold':float(best_threshold),'development_calibrated_metrics':best_calibrated,'context_ablation':ablations,'class_weights':class_weights.cpu().tolist(),'trainable_parameters':trainable,'total_parameters':total}
(OUT/'development_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')



## 8. Predict test cases and create the required submission ZIP

The test rows were loaded using only `id` and `text`; all other fields were discarded before training or prediction. Predictions follow the provided brief's format: one list entry per test case, with integer `pred` in `{0,1}`. The ZIP contains exactly one root-level `prediction.json`.


In [ ]:

test_logits,test_prob,test_pred=predict_windows(TEST_ROWS,TEST_WINDOWS,threshold=float(best_threshold))
prediction=[{'id':row['id'],'pred':int(pred)} for row,pred in zip(TEST_ROWS,test_pred)]
assert len(prediction)==len(TEST_ROWS)
assert [x['id'] for x in prediction]==[x['id'] for x in TEST_ROWS]
assert all(type(x['pred']) is int and x['pred'] in (0,1) for x in prediction)
pred_path=OUT/'prediction.json'; pred_path.write_text(json.dumps(prediction,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=OUT/'phase4_v4_lora_submission.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z: z.write(pred_path,arcname='prediction.json')
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist()==['prediction.json']; assert z.testzip() is None
    roundtrip=json.loads(z.read('prediction.json'))
assert roundtrip==prediction
print('Created:',pred_path,'and',zip_path)
print('Test predictions:',len(prediction),'predicted class counts:',dict(Counter(x['pred'] for x in prediction)))
print('Development metrics:',json.dumps(summary,indent=2))


## What this V4 stage establishes

This notebook evaluates one V4 hybrid + LoRA candidate using the trained Phase 1 checkpoint. It does not rerun language-model training. The prediction ZIP is an experimental candidate; compare its development Macro-F1, class-wise results, threshold, and eventual official score with the V1, V2, and V3 candidates before promotion. No public score is known until submitted.